# Finger Lakes hyperspectral surface analysis

**Author:** Caroline Kamal  
**Region:** Finger Lakes, New York  
**Mode:** Explicitly synthetic demonstration; these are not NASA observations or verified mineral maps.

This notebook walks through regional configuration, spectral preprocessing, PCA, k-means clustering, and conservative exposed-surface interpretation.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from upstate_hyperspectral.analysis import analyze_scene, build_summary
from upstate_hyperspectral.pipeline import run_demo
from upstate_hyperspectral.regions import get_region
from upstate_hyperspectral.synthetic import generate_demo_scene

## 1. Define the study area

The region includes Geneva, Seneca Lake, Cayuga Lake, Ithaca, and the surrounding agricultural corridor. Coordinates are WGS84 longitude/latitude.

In [ ]:
region = get_region("finger-lakes")
print(region.name)
print("Bounding box:", region.bbox)
print(region.description)

## 2. Generate the explicitly synthetic demonstration

The demonstration has the same 285-band dimensionality and approximate wavelength range as EMIT L2A, but its reflectance values and landscape are simulated.

In [ ]:
scene = generate_demo_scene(region, seed=2026)
print(scene.provenance)
print("Cube shape:", scene.shape)
print("Usable bands:", scene.good_wavelengths.sum())

## 3. Run PCA, k-means, and exposed-surface screening

Water and vegetation are excluded from mineral-related absorption proxies. Surface clusters should not be interpreted as confirmed mineral species or crop-disease diagnoses.

In [ ]:
result = analyze_scene(scene, n_clusters=6)
summary = build_summary(scene, result)
for key in ("valid_pixels", "water_pixels", "vegetation_pixels", "exposed_surface_pixels", "silhouette_score"):
    print(f"{key}: {summary[key]}")

## 4. Generate portfolio-ready figures

In [ ]:
output_dir = Path("../outputs/notebook-finger-lakes")
summary = run_demo(region, output_dir)
display(Image(filename=str(output_dir / "figures/study-area-overview.png")))

## 5. Search for actual NASA observations

Install the optional NASA dependencies and create a NASA Earthdata account before downloading. EMIT coverage of upstate New York is not guaranteed.

```python
from upstate_hyperspectral.nasa import search_emit_granules

results = search_emit_granules(region, "2023-05-01", "2026-08-25")
print(f"Observed EMIT granules: {len(results)}")
```